# Decide-then-Write: LFM3 + attached classifiers

The same Decide heads on LFM3-1.7B (LiquidAI/LFM3-1.7B, `trust_remote_code`), two variants: heads only on the frozen model, and LoRA (r=16 on attention, short-conv and MLP projections) + heads. Exports the merged backbone as GGUF Q4_0 for llama.cpp (`--embedding --pooling none`) and the heads. LFM Open License v1.0.

In [ ]:
# Setup: clone the experiment branch and install dependencies
import os, subprocess
REPO = "github.com/AwaisSDev/Codeply-Craft"
BRANCH = "codeply-li-dtw-experiment"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = f"https://{token}@{REPO}"
except Exception:
    url = f"https://{REPO}"
if not os.path.exists("/kaggle/working/codeply"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/codeply"], check=True)
os.chdir("/kaggle/working/codeply/research/dtw")
subprocess.run("pip -q install 'transformers>=4.48' peft accelerate sentencepiece", shell=True, check=True)
subprocess.run("cd data/out && for f in *.gz; do gunzip -kf $f; done", shell=True, check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# llama.cpp at the pinned commit: GGUF converter and quantizer
import os, subprocess
if not os.path.exists("/kaggle/working/llama.cpp"):
    subprocess.run("git clone https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp && cd /kaggle/working/llama.cpp && git checkout aa588a3463357ee990b314f41dda242869130704", shell=True, check=True)
    subprocess.run("cd /kaggle/working/llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF > /dev/null && cmake --build build -j4 --target llama-quantize > /dev/null", shell=True, check=True)
    subprocess.run("pip -q install -e /kaggle/working/llama.cpp/gguf-py", shell=True, check=True)

def export_gguf(run):
    """backbone -> F16 GGUF -> Q8_0 and Q4_0; heads -> heads.bin/json."""
    subprocess.run(f"python /kaggle/working/llama.cpp/convert_hf_to_gguf.py {run}/backbone --outtype f16 --outfile {run}/backbone-f16.gguf", shell=True, check=True)
    for q in ("Q8_0", "Q4_0"):
        subprocess.run(f"/kaggle/working/llama.cpp/build/bin/llama-quantize {run}/backbone-f16.gguf {run}/backbone-{q.lower()}.gguf {q}", shell=True, check=True)
    subprocess.run(f"python model/export_heads.py {run}", shell=True, check=True)

In [ ]:
import subprocess
COMMON = ("--model LiquidAI/LFM3-1.7B --causal --trust-remote-code --train data/out/train.spans.jsonl "
          "--val data/out/val.spans.jsonl --epochs 2 --bs 8 --val-limit 400")
for name, extra in {"lfm3-heads-frozen": "--freeze-backbone --head-lr 1e-3",
                    "lfm3-heads-lora": "--lora 16 --lr 2e-4 --head-lr 1e-3"}.items():
    for seed in [1, 2, 3]:
        run = f"runs/{name}-s{seed}"
        if os.path.exists(f"{run}/config.json"):
            continue
        subprocess.run(f"python model/train_encoder.py {COMMON} {extra} --seed {seed} --out {run}", shell=True, check=True)
        export_gguf(run)

In [ ]:
import json, glob
for c in sorted(glob.glob("runs/*/config.json")):
    print(c.split("/")[1], json.load(open(c))["val"])

In [ ]:
!cd runs && zip -qr /kaggle/working/dtw-lfm3-heads.zip . -x "*/backbone/*" "*/adapter/*" "*/backbone-f16.gguf"
!ls -la /kaggle/working/dtw-lfm3-heads.zip